# Laser Head Camera

This notebook uses the Poetry-managed environment. All dependencies from `pyproject.toml` are available.


In [ ]:
# Verify we're using the correct environment
import sys
print(f"Python: {sys.executable}")


In [ ]:
from femtum_sdk.api.contracts.v1.hardware.health_pb2 import HealthRequest
from femtum_sdk.api import HardwareApi

api = HardwareApi(host="172.26.6.110:5210")
api.__enter__()
health_response = api.health.GetHealth(HealthRequest())
print(health_response)


## Capture Camera Image

In [ ]:
from femtum_sdk.api.contracts.v1.dto.image_pb2 import Image, ImageFormat, ImageSize
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import DistanceMeasure, DistanceMeasureUnit, EnergyMeasure, EnergyMeasureUnit
from femtum_sdk.api.contracts.v1.hardware.camera_pb2 import CaptureImageRequest
from femtum_sdk.api.contracts.v1.hardware.laser_execution_pb2 import LaserExecutionDistance, LaserExecutionOptions, SimulateLaserExecutionRecipeResponse
import io
from PIL import Image

image: Image = api.laser_head_camera.CaptureImage(CaptureImageRequest(
    Format=ImageFormat.IMAGE_FORMAT_PNG, 
    Size=ImageSize(
        Width=4512, 
        Height=4512
    )
))

Image.open(io.BytesIO(image.Data)).show()


## Get Camera Infos

In [ ]:
from femtum_sdk.api.contracts.v1.hardware.camera_pb2 import ReadCameraInfoRequest


camera_info = api.laser_head_camera.ReadCameraInfo(ReadCameraInfoRequest())
print(camera_info)

## Set Camera ROI

In [ ]:
from femtum_sdk.api.contracts.v1.dto.image_pb2 import CenteredRegionOfInterest


api.laser_head_camera.UpdateROI(CenteredRegionOfInterest(
    Width=2048,
    Height=2048
))

image: Image = api.laser_head_camera.CaptureImage(CaptureImageRequest(
    Format=ImageFormat.IMAGE_FORMAT_PNG, 
    Size=ImageSize(
        Width=1024, 
        Height=1024
    )
))

Image.open(io.BytesIO(image.Data)).show()

# Reset to Default ROI
api.laser_head_camera.UpdateROI(CenteredRegionOfInterest(
    Width=4512,
    Height=4512
))


 ## Reset Camera ROI

In [ ]:
from femtum_sdk.api.contracts.v1.hardware.camera_pb2 import ResetROIRequest


api.laser_head_camera.ResetROI(ResetROIRequest())

# Set image parameters

In [ ]:
from femtum_sdk.api.contracts.v1.dto.camera_pb2 import CameraImageParameters, ExposureTime, Gain, ManualExposureTime, ManualGain
from femtum_sdk.api.contracts.v1.dto.image_pb2 import ImageFormat, ImageSize
from femtum_sdk.api.contracts.v1.hardware.camera_pb2 import CaptureImageRequest

from google.protobuf.duration_pb2 import Duration

from PIL import Image

import io

api.laser_head_camera.UpdateImageParameters(CameraImageParameters(
    ExposureTime=ExposureTime(
        Manual=ManualExposureTime(
            Value=Duration(nanos=100000)
        ),
    ),
    Gain=Gain(
        Manual=ManualGain(
            Value=5.0
        ),
    ),
    Gamma=1.0,
    ContrastPercent=0.0,
    BrightnessPercent=0.0,
))


image: Image = api.laser_head_camera.CaptureImage(CaptureImageRequest(
    Format=ImageFormat.IMAGE_FORMAT_PNG, 
    Size=ImageSize(
        Width=1024, 
        Height=1024
    )
))

Image.open(io.BytesIO(image.Data)).show()


# Reset image parameters

In [ ]:
from femtum_sdk.api.contracts.v1.hardware.camera_pb2 import ResetImageParametersRequest

api.laser_head_camera.ResetImageParameters(ResetImageParametersRequest())

## Cleanup

In [ ]:
api.__exit__()